In [1]:
import numpy as np
import pandas as pd
from sklearn import datasets, ensemble
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.utils.fixes import parse_version
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
%matplotlib inline
import matplotlib.pyplot as plt

In [2]:
"""
Load the days, add two new columns given the timestamp, hours and days
"""
df = pd.read_excel("../../data/data_Hw3_new.xlsx")

df['timestamp'] = pd.to_datetime(df['timestamp'])

df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour

feature_cols = ['A0', 'A1', 'A2', 'A3', 'temperature', 'humidity']
target_col = 'O3'

"""
Count the amount of hours in each of the days, remove any of them in case it does not have the 24 hours (none in our case)
"""
dates = df.groupby('date').size()
complete_dates = dates[dates == 24].index
print(dates[dates!= 24])
df = df[df['date'].isin(complete_dates)].copy()
df.head()

Series([], dtype: int64)


,timestamp,A0,A1,A2,A3,temperature,humidity,O3,date,hour
0,2023-06-04 00:00:00,455.58,451.83,449.58,445.33,21.47,69.58,54.0,2023-06-04,0
1,2023-06-04 01:00:00,452.92,452.75,448.75,446.25,21.17,71.98,64.0,2023-06-04,1
2,2023-06-04 02:00:00,453.50,452.58,444.92,447.00,21.03,71.65,56.0,2023-06-04,2
3,2023-06-04 03:00:00,450.75,452.25,437.58,446.50,21.10,70.75,43.0,2023-06-04,3
4,2023-06-04 04:00:00,448.58,451.50,433.33,445.50,20.79,69.68,34.0,2023-06-04,4


In [3]:
"""
Sort the days and divide them between train (80 days) and testing
"""
days = 80
testing_days = len(complete_dates) - days

hours = days*24
df = df.sort_values(by=["date", "hour"]).reset_index(drop=True)

train = df[:hours]
test = df[hours:]
train_x = train[feature_cols]
train_y = train[target_col]
test_x = test[feature_cols]
test_y = test[target_col]

In [4]:
"""
Train the best possible model using Gradient Boosting
"""
param_grid = {'max_depth': list(range(1, 15))}
gbr = ensemble.GradientBoostingRegressor(n_estimators = 500, min_samples_split = 5, learning_rate = 0.01, random_state=0)

grid = GridSearchCV(gbr, param_grid, cv=5, scoring='r2', n_jobs=-1, return_train_score=True)
grid.fit(train_x, train_y)

print("Best params:",)
print("Best CV r2:", grid.best_score_)

best_model = grid.best_estimator_
# final evaluation on the untouched test set:
test_r2 = best_model.score(test_x, test_y)
print("Test R2:", test_r2)

Best params:
Best CV r2: 0.7334105921326328
Test R2: 0.8529664096342782


In [5]:
"""
Add the predicted data to the dataset
"""

df["predicted"] = best_model.predict(df[feature_cols])

# Update the views
train = df[:hours]
test = df[hours:]
train_x = train[feature_cols]
train_y = train[target_col]
test_x = test[feature_cols]
test_y = test[target_col]


# Expected shape: R(N, M), (24, 80)
L = np.array(train.pivot(index='hour', columns='date', values="predicted").sort_index().values)
print("L shape:", L.shape)

# Expected shape: R(N, 1), (24, 1) 
Lambda = L.mean(axis=1, keepdims=True)
print("Lambda shape:", Lambda.shape)

# Expected shape: R(N, M), (24, 80)
X = L - Lambda
print("X shape:", X.shape)

#Ensuring X is calculated correctly
for i in range(len(X.T)):
    if not np.all(X[:, i] == (L[:, i] - Lambda[:, 0])):
        print("ERROR!!!")

        
# Expected shape: 
# U: R(N, N), (24, 24)
# S: R(min(N, M)), (min(24, 80)), (24)
# Vt: R(M, M), (80, 80)
U, S, Vt = np.linalg.svd(X, full_matrices=True)
print("---SVD shape---")
print("U shape:", U.shape)
print("S shape:", S.shape)
print("Vt shape:", Vt.shape)

L shape: (24, 80)
Lambda shape: (24, 1)
X shape: (24, 80)
---SVD shape---
U shape: (24, 24)
S shape: (24,)
Vt shape: (80, 80)


In [6]:
"""
Find the best RMSE compared to the one obtained when not denoising
"""

RMSE_k = []
R2_k = []

# Matrice with the predicted values for the testing days
# Expected shape: R(N, test_days), R(24, 51)
M_test = np.array(test['predicted'].values.reshape(testing_days, 24)).T
print("M_test shape:", M_test.shape)

# Real values
# Expected shape: R(test_days * 24), R(1224)
y_test = test[target_col]
print("Y_test shape:", y_test.shape)


for k in range(1, 25):
    # Expected shape: R(N, k), (24, k)
    U_r = U[:, :k]
    if U_r.shape != (24, k):
        print("Shape not correct")

    hat_x = M_test - Lambda
    M_denoised = Lambda + (U_r @ (U_r.T @ hat_x))
    y_denoised = M_denoised.flatten(order='f')

    RMSE_k.append(np.sqrt(mean_squared_error(y_test, y_denoised)))
    R2_k.append(r2_score(y_test, y_denoised))
                
print("Original RMSE:", RMSE_k[-1], " Best RMSE:", min(RMSE_k), "Index:" , np.argmin(RMSE_k))
print("Original R2:", R2_k[-1], " Best R2:", max(R2_k), "Index:", np.argmax(R2_k))

M_test shape: (24, 51)
Y_test shape: (1224,)
Original RMSE: 7.894792067829645  Best RMSE: 7.880346320389621 Index: 22
Original R2: 0.8529664096342784  Best R2: 0.8535039961484735 Index: 22


In [7]:
"""
Add the predicted data to the dataset
"""

df["predicted"] = best_model.predict(df[feature_cols])

# Update the views
train = df[:hours]
test = df[hours:]
train_x = train[feature_cols]
train_y = train[target_col]
test_x = test[feature_cols]
test_y = test[target_col]


# Expected shape: R(N, M), (24, 80)
L_ref = np.array(train.pivot(index='hour', columns='date', values=target_col).sort_index().values)
print("L_ref shape:", L_ref.shape)

# Expected shape: R(N, 1), (24, 1) 
Lambda_ref = L_ref.mean(axis=1, keepdims=True)
print("Lambda_ref shape:", Lambda_ref.shape)

# Expected shape: R(N, M), (24, 80)
X_ref = L_ref - Lambda_ref
print("X_ref shape:", X_ref.shape)

#Ensuring X is calculated correctly
for i in range(len(X_ref.T)):
    if not np.all(X_ref[:, i] == (L_ref[:, i] - Lambda_ref[:, 0])):
        print("ERROR!!!")

        
# Expected shape: 
# U: R(N, N), (24, 24)
# S: R(min(N, M)), (min(24, 80)), (24)
# Vt: R(M, M), (80, 80)
U_ref, S_ref, Vt_ref = np.linalg.svd(X_ref, full_matrices=True)
print("---SVD shape---")
print("U_ref shape:", U_ref.shape)
print("S_ref shape:", S_ref.shape)
print("Vt_ref shape:", Vt_ref.shape)

L_ref shape: (24, 80)
Lambda_ref shape: (24, 1)
X_ref shape: (24, 80)
---SVD shape---
U_ref shape: (24, 24)
S_ref shape: (24,)
Vt_ref shape: (80, 80)


In [8]:
"""
Find the best RMSE compared to the one obtained when not denoising
"""

RMSE_k_ref = []
R2_k_ref = []

# Matrice with the predicted values for the testing days
# Expected shape: R(N, test_days), R(24, 51)
M_test = np.array(test['predicted'].values.reshape(testing_days, 24)).T
print("M_test shape:", M_test.shape)

# Real values
# Expected shape: R(test_days * 24), R(1224)
y_test = test[target_col]
print("Y_test shape:", y_test.shape)


for k in range(1, 25):
    # Expected shape: R(N, k), (24, k)
    U_r = U_ref[:, :k]
    if U_r.shape != (24, k):
        print("Shape not correct")

    hat_x = M_test - Lambda_ref
    M_denoised = Lambda_ref + (U_r @ (U_r.T @ hat_x))
    y_denoised = M_denoised.flatten(order='f')

    RMSE_k_ref.append(np.sqrt(mean_squared_error(y_test, y_denoised)))
    R2_k_ref.append(r2_score(y_test, y_denoised))
                
print("Original RMSE:", RMSE_k_ref[-1], " Best RMSE:", min(RMSE_k_ref), "Index:" , np.argmin(RMSE_k_ref))
print("Original R2:", R2_k_ref[-1], " Best R2:", max(R2_k_ref), "Index:", np.argmax(R2_k_ref))

M_test shape: (24, 51)
Y_test shape: (1224,)
Original RMSE: 7.89479206782965  Best RMSE: 7.863517313515285 Index: 22
Original R2: 0.8529664096342782  Best R2: 0.8541290320698314 Index: 22
